In [1]:
import sys
import os
import pandas as pd
import geopandas as gpd
import shapely
import tqdm
from datetime import datetime 
sys.path.append("../../../quetzal")

from quetzal.model import stepmodel
from quetzal.io.gtfs_reader import importer

# 0. Params

In [2]:
start_time = '06:00:00'
end_time = '10:00:00'

# 1. PT except train

In [3]:
ipath = '../../inputs/pt/raw/gtfs/'

In [4]:
list_files = os.listdir(ipath)
file = list_files[1]
path_file = ipath + file

print(path_file)

../../inputs/pt/raw/gtfs/bordeaux.gtfs.zip


In [5]:
all_links = []
all_nodes = []

In [6]:
mode_labels = {
    0: 'tramway',
    2: 'rail',
    3: 'bus',
    712: 'schoolbus',
    4: 'ferry'
}

In [7]:
def time_seconds(time_str):
    return int(time_str.split(':')[0])*3600 + int(time_str.split(':')[1])*60 + int(time_str.split(':')[2])*1

In [8]:
for i, file in enumerate(list_files):
    path_file = ipath + file
    print(path_file)

    feed = importer.GtfsImporter(path=path_file, dist_units='m')
    if '20260623' in feed.get_dates():
        feed = feed.clean().restrict_to_dates(['20260623'])
    else:
        feed = feed.clean().restrict_to_dates(['20260915'])

    group_shapes = feed.shapes.copy()
    group_shapes['point'] = group_shapes.apply(lambda x: shapely.geometry.Point(x['shape_pt_lon'], x['shape_pt_lat']), 1)
    group_shapes = group_shapes.groupby('shape_id').agg({'point': shapely.geometry.LineString, 'shape_pt_sequence': 'max'})
    group_shapes_ = group_shapes.copy()
    group_shapes_.reset_index(inplace=True)
    group_shapes_['point'] = group_shapes_['point'].apply(shapely.to_wkt)
    grouped_shapes = group_shapes_.groupby(['point', 'shape_pt_sequence']).agg({'shape_id': list}).reset_index()['shape_id'].to_list()

    assigned = {}
    for l in grouped_shapes:
        for x in l:
            assigned[x] = l[0]

    stop_times_with_shapes = feed.stop_times.copy().merge(feed.trips.copy()[['trip_id', 'route_id', 'shape_id']], on='trip_id').merge(feed.routes.copy()[['route_id', 'route_type', 'route_long_name']])
    stop_times_with_shapes['departure_time_sec'] = stop_times_with_shapes['departure_time'].map(time_seconds)
    stop_times_with_shapes['shape_id'] = stop_times_with_shapes['shape_id'].map(assigned)
    stop_times_with_shapes = stop_times_with_shapes.sort_values(['shape_id', 'stop_sequence', 'departure_time_sec'])
    stop_times_with_shapes = stop_times_with_shapes[['stop_id', 'stop_sequence', 'shape_id', 'route_type', 'departure_time', 'arrival_time']]
    stop_times_with_shapes = stop_times_with_shapes.groupby(['stop_id', 'stop_sequence', 'shape_id', 'route_type']).agg({'departure_time': list, 'arrival_time': list})#.sort_values(['shape_id', 'stop_sequence']).reset_index()
    stop_times_with_shapes = stop_times_with_shapes.reset_index()

    departures = stop_times_with_shapes.rename(columns={'departure_time': 'departures', 'stop_id': 'a'})[['a', 'shape_id', 'route_type', 'departures']]
    arrivals = stop_times_with_shapes.rename(columns={'arrival_time': 'arrivals', 'stop_id': 'b'})[['b', 'shape_id', 'route_type', 'arrivals']]

    try:
        built = feed.build(date = '20260623', time_range = [start_time, end_time])
    except IndexError:
        built = feed.build(date = '20260915', time_range = [start_time, end_time])

    built.links.drop(columns=['departure_time', 'arrival_time'], inplace=True)
    built.links = built.links.merge(departures, on=['a', 'shape_id', 'route_type'], how='left')
    built.links = built.links.merge(arrivals, on=['b', 'shape_id', 'route_type'], how='left')
    
    built.links['agency_name'] = built.links['agency_id'].map(feed.agency.set_index('agency_id')['agency_name'].to_dict())

    columns = ['geometry', 'drop_off_type', 'route_id_karouest', 'route_id',
       'route_long_name', 'route_type', 'length', 'link_sequence', 'b',
       'agency_id', 'trip_id', 'time', 'a', 'headway', 'route_width',
       'trip_id_karouest', 'direction_id', 'route_short_name', 'route_color', 'pickup_type'] + ['departures', 'arrivals']

    built.links['route_id_orig']  = built.links['route_id'].copy()
    built.links['route_short_name_orig']  = built.links['route_id'].copy()
    built.links['trip_id_orig']  = built.links['trip_id'].copy()
    built.links['route_id'] = built.links['route_short_name'] + '_' + built.links['agency_name'] + '_' + built.links['route_type'].astype(str)
    built.links['trip_id'] = built.links.apply(lambda x: x['trip_id_orig'].replace(x['route_id_orig'], x['route_id']), 1)

    built.links['drop_off_type'] = None
    built.links['length'] = built.links.length
    built.links['route_width'] = 1
    built.links['pickup_type'] = None

    built.nodes['index'] = 'node_' + str(i) + '_' + built.nodes['stop_id']

    stop_to_node = built.nodes.set_index('stop_id')['index'].to_dict()
    built.links['a'] = built.links['a'].map(stop_to_node)
    built.links['b'] = built.links['b'].map(stop_to_node)
    built.links['index'] = 'link_' + str(i) + '_' + built.links.index.astype(str)

    try:
        built.nodes = built.nodes.drop(columns=['location_type', 'parent_station', 'platform_code',
            'stop_desc', 'stop_timezone', 'stop_url', 'wheelchair_boarding', 'zone_id',])
    except KeyError:
        built.nodes = built.nodes.drop(columns=['location_type', 'parent_station', 'stop_url', 'wheelchair_boarding', 'zone_id',])
    relevant_nodes = set(built.links['a'].values.tolist() + built.links['b'].values.tolist())
    built.nodes = built.nodes.loc[built.nodes['index'].isin(relevant_nodes)]
    
    built.links['route_type'] = built.links['route_type'].map(mode_labels)
    built.nodes.set_index('index', inplace=True)
    
    links = built.links.copy()
    nodes = built.nodes.copy()

    links = links.to_crs(epsg=4326).set_index('index')

    node_list = list(set(links['a'].values.tolist() + links['b'].values.tolist()))
    nodes = nodes.loc[node_list]
    nodes = nodes.to_crs(epsg=4326)

    all_nodes.append(nodes)
    all_links.append(links)

../../inputs/pt/raw/gtfs/bac_gironde-aggregated-gtfs.zip
Restricting to date…
Grouping services…
Building patterns…
Converting to frequencies…
Building links and nodes…
export geometries in epsg: 32630
../../inputs/pt/raw/gtfs/bordeaux.gtfs.zip
Restricting to date…
Grouping services…
Restricting to date…
Grouping services…
Building patterns…
Converting to frequencies…
Building links and nodes…
export geometries in epsg: 32630
../../inputs/pt/raw/gtfs/ca_du_libournais-aggregated-gtfs.zip
Restricting to date…
Grouping services…
Restricting to date…
Grouping services…
Building patterns…
Converting to frequencies…
Building links and nodes…
export geometries in epsg: 32630
../../inputs/pt/raw/gtfs/cc_sud_gironde-aggregated-gtfs.zip
Restricting to date…
Grouping services…
Building patterns…
Converting to frequencies…
Building links and nodes…
export geometries in epsg: 32630
../../inputs/pt/raw/gtfs/cobas-aggregated-gtfs.zip
Restricting to date…
Grouping services…
Building patterns…
Converti

In [9]:
nodes_pt = pd.concat(all_nodes)
nodes_pt = gpd.GeoDataFrame(nodes_pt, geometry='geometry', crs='EPSG:4326')

links_pt = pd.concat(all_links)
links_pt = gpd.GeoDataFrame(links_pt, geometry='geometry', crs='EPSG:4326')

In [10]:
### Remove school buses
links_pt = links_pt.loc[links_pt['route_type'] != 'schoolbus']
links_pt = links_pt.loc[links_pt['route_long_name'].apply(lambda x: 'SCODI' not in x)]
nodes_pt = nodes_pt.loc[list(set(links_pt['a'].values.tolist() + links_pt['b'].values.tolist()))]

# 2. Rail

In [11]:
ipath = '../../inputs/pt/raw/train/'
gironde_perimeter = gpd.GeoDataFrame({'geometry': [shapely.union_all(gpd.read_file('../../inputs/zones/zones_gironde.gpkg').to_crs(epsg=4326)['geometry'].values.tolist())]},
                                     geometry='geometry',
                                     crs='EPSG:4326')

In [12]:
path_file = ipath + os.listdir(ipath)[0]
print(path_file)

../../inputs/pt/raw/train/Export_OpenData_SNCF_GTFS_NewTripId (1).zip


In [13]:
mode_labels = {
    0: 'tramway',
    2: 'rail',
    3: 'bus'
}

In [14]:
def time_seconds(time_str):
    return int(time_str.split(':')[0])*3600 + int(time_str.split(':')[1])*60 + int(time_str.split(':')[2])*1

In [15]:
feed = importer.GtfsImporter(path=path_file, dist_units='m')
feed = feed.clean()

In [16]:
try:
    built = feed.build(date = '20260623', time_range = [start_time, end_time])
except IndexError:
    built = feed.build(date = '20260915', time_range = [start_time, end_time])

Restricting to date…
Grouping services…
Restricting to date…
Grouping services…
Building patterns…
Converting to frequencies…
Building links and nodes…
export geometries in epsg: 32631


In [17]:
def seconds_to_time(seconds):
    if pd.isna(seconds):
        return None
    seconds = int(seconds)
    h = seconds // 3600
    m = (seconds % 3600) // 60
    s = seconds % 60
    return f"{h:02d}:{m:02d}:{s:02d}"

built.links['departures'] = built.links['departure_time'].apply(lambda x: [seconds_to_time(x)])
built.links['arrivals'] = built.links['arrival_time'].apply(lambda x: [seconds_to_time(x)])

In [18]:
columns = ['geometry', 'drop_off_type', 'route_id_karouest', 'route_id',
       'route_long_name', 'route_type', 'length', 'link_sequence', 'b',
       'agency_id', 'trip_id', 'time', 'a', 'headway', 'route_width',
       'trip_id_karouest', 'direction_id', 'route_short_name', 'route_color', 'pickup_type'] + ['departures', 'arrivals']

In [19]:
built.links['route_id_orig']  = built.links['route_id'].copy()
built.links['trip_id_orig']  = built.links['trip_id'].copy()
built.links['route_id'] = built.links['route_short_name']
built.links['trip_id'] = built.links.apply(lambda x: x['trip_id_orig'].replace(x['route_id_orig'], x['route_id']), 1)

In [20]:
built.links['drop_off_type'] = None
built.links['length'] = built.links.length
built.links['route_width'] = 1
built.links['pickup_type'] = None

In [21]:
built.nodes['index'] = 'node_' + built.nodes['stop_id']

In [22]:
stop_to_node = built.nodes.set_index('stop_id')['index'].to_dict()
built.links['a'] = built.links['a'].map(stop_to_node)
built.links['b'] = built.links['b'].map(stop_to_node)

In [23]:
built.links['index'] = 'link_' + built.links.index.astype(str)

In [24]:
built.nodes = built.nodes.drop(columns=['location_type', 'parent_station',
        'stop_desc', 'stop_url', 'zone_id',])

In [25]:
relevant_nodes = set(built.links['a'].values.tolist() + built.links['b'].values.tolist())
built.nodes = built.nodes.loc[built.nodes['index'].isin(relevant_nodes)]

In [26]:
built.links['route_type'] = built.links['route_type'].map(mode_labels)

In [27]:
built.nodes.set_index('index', inplace=True)

In [28]:
## Isolate bus links :
links = built.links.copy()
nodes = built.nodes.copy()

links = links.to_crs(epsg=4326).set_index('index')

node_list = list(set(links['a'].values.tolist() + links['b'].values.tolist()))
nodes = nodes.loc[node_list]
nodes = nodes.to_crs(epsg=4326)

In [29]:
nodes_gironde = nodes.sjoin(gironde_perimeter, predicate='intersects').reset_index()['index'].values.tolist()
links_rail = links.loc[links['a'].isin(nodes_gironde) & links['b'].isin(nodes_gironde)]
nodes_rail = nodes.loc[nodes_gironde]

# 3. Aggregate

In [30]:
opath = ipath = '../../inputs/pt/'

In [31]:
links = pd.concat([links_pt, links_rail])
nodes = pd.concat([nodes_pt, nodes_rail])

In [32]:
links['speed'] = (links['length']/links['time'])*3.6

In [33]:
links = links.loc[(~links['departures'].isna()) & (~links['arrivals'].isna())]
nodes = nodes.loc[list(set(links['a']).union(set(links['b'])))]
nodes = nodes.set_geometry('geometry')

In [34]:
nodes = nodes.sjoin(gironde_perimeter, predicate='intersects').drop(columns='index_right')
links = links.loc[links['a'].isin(nodes.index) & links['b'].isin(nodes.index)]

In [35]:
links.to_file(opath + 'links.geojson')
nodes.to_file(opath + 'nodes.geojson')

In [36]:
end_of_notebook

NameError: name 'end_of_notebook' is not defined

In [ ]:
links.to_file(opath + 'quenedi/links.geojson')
nodes.to_file(opath + 'quenedi/nodes.geojson')

In [ ]:
links['departure_times'] = links['departures'].apply(lambda x: [time_seconds(y) for y in x])
links['arrival_times'] = links['arrivals'].apply(lambda x: [time_seconds(y) for y in x])

In [ ]:
links_no_geom = links.drop(columns=['departure_time', 'arrival_time', 'geometry']).explode(['departure_times', 'arrival_times']).rename(columns={'departure_times': 'departure_time', 'arrival_times': 'arrival_time'}).reset_index()
links_no_geom['journey'] = links_no_geom.groupby('index').cumcount().astype(str)
links_no_geom['index_'] = links_no_geom['index'] + '_' + links_no_geom['journey']
links_no_geom['trip_id'] += '_' + links_no_geom['journey']
links = gpd.GeoDataFrame(links_no_geom.merge(links.reset_index()[['index', 'geometry']]), geometry='geometry', crs='EPSG:4326')

In [ ]:
links.to_file(opath + 'links.geojson')
nodes.to_file(opath + 'nodes.geojson')